Phase 2: Data Refinement & Enrichment (Silver Layer)
The Goal: Transform raw technical logs into a structured, human-readable format that is ready for analysis.
What’s happening here:
Time-Series Alignment: I'm converting raw UNIX timestamps into human-readable dates. This is to observe trends around peak viewing hours or seasonality.
Exploratory Profiling: a high-level inventory of the data so stakeholders know the boundaries of what they are looking at. (NB: because the sample dataset is so sparse, I'm going to be generating synthetic data in the gold stage to have a more interesting dataset to work with.)
Data Flattening (The Join & Explode): For granular reporting, I'm linking my content metadata (Movies) with my engagement logs (Ratings). "Exploding" genres ensures a movie listed under both "Sci-Fi" and "Action" is accurately counted in both categories (in our source data, a movie could have multiple genres listed in a single record, which is not practical for reporting). Bonus! because genre can be so nuanced (eg, Action Comedy, Comedy Horror, RomCom), flattening this data supports cross-category analysis (eg, what kinds of comedy are most popular over the summer? late at night? when some particular viral comic content has been trending?). 

In [153]:
import pandas as pd
ratings = pd.read_parquet("../data/02_bronze/ratings.parquet")
movies = pd.read_parquet("../data/02_bronze/movies.parquet")
tags = pd.read_parquet("../data/02_bronze/tags.parquet")


In [154]:
# Convert timestamps
ratings["rating_timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")


ratings.drop(columns=["timestamp"], inplace=True)

pd.set_option('display.width', 300)
print(ratings.head())


   userId  movieId  rating                     ingestion_ts  source_file    rating_timestamp
0       1       31     2.5 2026-05-06 01:50:57.706686+00:00  ratings.csv 2009-12-14 02:52:24
1       1     1029     3.0 2026-05-06 01:50:57.706686+00:00  ratings.csv 2009-12-14 02:52:59
2       1     1061     3.0 2026-05-06 01:50:57.706686+00:00  ratings.csv 2009-12-14 02:53:02
3       1     1129     2.0 2026-05-06 01:50:57.706686+00:00  ratings.csv 2009-12-14 02:53:05
4       1     1172     4.0 2026-05-06 01:50:57.706686+00:00  ratings.csv 2009-12-14 02:53:25


In [155]:
# investigating some basic stats

# how many unique users?
print(f"Unique users in ratings: {ratings['userId'].nunique()}")
print(f"Unique users in tags: {tags['userId'].nunique()}")

# how many unique movies?
print(f"Unique movies in ratings: {ratings['movieId'].nunique()}")
print(f"Unique movies in movies: {movies['movieId'].nunique()}")
print(f"Unique movies in tags: {tags['movieId'].nunique()}")

#what date range do the ratings cover?
print(f"Ratings date range: {ratings['rating_timestamp'].min()} to {ratings['rating_timestamp'].max()}")

Unique users in ratings: 671
Unique users in tags: 61
Unique movies in ratings: 9066
Unique movies in movies: 9125
Unique movies in tags: 689
Ratings date range: 1995-01-09 11:46:49 to 2016-10-16 17:57:24


In [156]:
# Join movies and ratings
silver_df = ratings.merge(movies, on="movieId")
silver_df.drop(columns=["ingestion_ts_x", "source_file_x", "ingestion_ts_y", "source_file_y"], inplace=True)


print(silver_df.head())



   userId  movieId  rating    rating_timestamp                                           title                            genres
0       1       31     2.5 2009-12-14 02:52:24                          Dangerous Minds (1995)                             Drama
1       1     1029     3.0 2009-12-14 02:52:59                                    Dumbo (1941)  Animation|Children|Drama|Musical
2       1     1061     3.0 2009-12-14 02:53:02                                 Sleepers (1996)                          Thriller
3       1     1129     2.0 2009-12-14 02:53:05                     Escape from New York (1981)  Action|Adventure|Sci-Fi|Thriller
4       1     1172     4.0 2009-12-14 02:53:25  Cinema Paradiso (Nuovo cinema Paradiso) (1989)                             Drama


In [157]:
# explode genres: normalize the genres field by exploding the pipe-delimited values into a genre dimension to enable downstream aggregation of engagement metrics by content category.

silver_df["genres"] = silver_df["genres"].str.split("|")
silver_df = silver_df.explode("genres")
print(silver_df.head())



   userId  movieId  rating    rating_timestamp                   title     genres
0       1       31     2.5 2009-12-14 02:52:24  Dangerous Minds (1995)      Drama
1       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)  Animation
1       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)   Children
1       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)      Drama
1       1     1029     3.0 2009-12-14 02:52:59            Dumbo (1941)    Musical


In [158]:
# save as parquet
silver_df.to_parquet("../data/03_silver/silver_interactions.parquet", index=False)